# NYC Yellow Taxi ETL
## Source Data Profiling

Purpose:
Understand the structure and quality of the raw NYC TLC January 2026 dataset before designing validation, cleaning, and transformation rules.

In [5]:
import pandas as pd

df = pd.read_csv("../data/raw/yellow_tripdata_2026-01.csv",low_memory=False)

In [23]:
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,7.2,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,7.9,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,10.7,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,38.7,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,13.5,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75


In [24]:
df.shape

(3724889, 20)

## Dataset Overview

The dataset contains **3,724,889 taxi trip records** with **20 columns**.

In [25]:
df.columns

Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'Airport_fee',
       'cbd_congestion_fee'],
      dtype='object')

In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3724889 entries, 0 to 3724888
Data columns (total 20 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   VendorID               int64  
 1   tpep_pickup_datetime   object 
 2   tpep_dropoff_datetime  object 
 3   passenger_count        float64
 4   trip_distance          float64
 5   RatecodeID             float64
 6   store_and_fwd_flag     object 
 7   PULocationID           int64  
 8   DOLocationID           int64  
 9   payment_type           int64  
 10  fare_amount            float64
 11  extra                  float64
 12  mta_tax                float64
 13  tip_amount             float64
 14  tolls_amount           float64
 15  improvement_surcharge  float64
 16  total_amount           float64
 17  congestion_surcharge   float64
 18  Airport_fee            float64
 19  cbd_congestion_fee     float64
dtypes: float64(13), int64(4), object(3)
memory usage: 568.4+ MB


### Observations

- The raw dataset contains **20 columns** describing each taxi trip.
- Most columns have appropriate data types, while the datetime columns are currently stored as strings (`object`) and will require conversion during the transformation stage.
- Columns containing missing values have been automatically assigned the `float64` data type by pandas. Whether these should remain nullable or be converted later will depend on the cleaning strategy.
- The dataset has been loaded successfully and no structural issues were observed during the initial inspection.
- At this stage, the dataset is intentionally left unchanged. The purpose of this notebook is to profile the raw data, not to modify it.

In [27]:
df.isnull().sum()

VendorID                       0
tpep_pickup_datetime           0
tpep_dropoff_datetime          0
passenger_count          1088058
trip_distance                  0
RatecodeID               1088058
store_and_fwd_flag       1088058
PULocationID                   0
DOLocationID                   0
payment_type                   0
fare_amount                    0
extra                          0
mta_tax                        0
tip_amount                     0
tolls_amount                   0
improvement_surcharge          0
total_amount                   0
congestion_surcharge     1088058
Airport_fee              1088058
cbd_congestion_fee             0
dtype: int64

In [28]:
df.duplicated().sum()

np.int64(0)

In [29]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
VendorID,3724889.0,1.873598,0.701458,1.00,2.0,2.00,2.00,7.00
passenger_count,2636831.0,1.256271,0.670243,0.00,1.0,1.00,1.00,9.00
trip_distance,3724889.0,6.455647,648.885528,0.00,1.0,1.81,3.73,269097.48
RatecodeID,2636831.0,5.218853,19.653699,1.00,1.0,1.00,1.00,99.00
PULocationID,3724889.0,161.437190,67.066574,1.00,114.0,161.00,233.00,265.00
DOLocationID,3724889.0,160.993570,71.036040,1.00,107.0,162.00,234.00,265.00
payment_type,3724889.0,0.846564,0.712049,0.00,0.0,1.00,1.00,4.00
fare_amount,3724889.0,20.804254,18.927007,-2555.20,10.0,15.60,26.10,2555.20
extra,3724889.0,1.023055,1.707322,-7.50,0.0,0.00,2.50,17.46
mta_tax,3724889.0,0.483377,0.114556,-0.50,0.5,0.50,0.50,4.75


In [32]:
df["VendorID"].value_counts(dropna=False)

VendorID
2    2965742
1     710425
7      44705
6       4017
Name: count, dtype: int64

In [33]:
df["payment_type"].value_counts(dropna=False)

payment_type
1    2249747
0    1088058
2     314043
4      56400
3      16641
Name: count, dtype: int64

In [34]:
df["RatecodeID"].value_counts(dropna=False)

RatecodeID
1.0     2390495
NaN     1088058
99.0     110864
2.0       83592
5.0       32030
3.0       11541
4.0        8304
6.0           5
Name: count, dtype: int64

In [35]:
df["store_and_fwd_flag"].value_counts(dropna=False)

store_and_fwd_flag
N      2634494
NaN    1088058
Y         2337
Name: count, dtype: int64

# Summary of Findings

Here's what stood out after taking a first look at the NYC Yellow Taxi (January 2026) dataset:

- The dataset has **3,724,889 trip records** and **20 columns** — a solid, realistic size for building an ETL pipeline on.
- It loaded cleanly, no structural problems.
- The pickup/dropoff datetime columns are stored as text (`object`) right now, so they'll need to be converted to actual datetime values during transformation.
- Five columns — `passenger_count`, `RatecodeID`, `store_and_fwd_flag`, `congestion_surcharge`, and `Airport_fee` — are all missing the exact same number of values. That's not random, so it's worth digging into before deciding how to handle it.
- No fully duplicated rows, so we don't need a dedup step for now.
- The summary stats show some weird stuff — trip distances that are way too large, and negative values in a few money columns. Need to check the official NYC TLC docs to figure out if these are real edge cases or just bad data.
- A few values in the categorical columns look off too — worth cross-checking against the data dictionary to make sure they're valid.

## Conclusion

This notebook was just about understanding the raw data before touching it — no cleaning, validation, or transforming has happened yet.

What we found here will shape the actual rules (validation, cleaning, transformation logic) for the next stage of the pipeline.